
<details>
<summary>**Set-Up**</summary>

1. Mount Google Drive
2. Install packages
3. Set video paths
4. Check GPU


</details>

In [ ]:
# ===== Cell 1 =====
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===== Cell 2 =====
!pip install -q opencv-python-headless pandas tqdm

In [ ]:
# ===== Cell 3 =====
from pathlib import Path

VIDEO_DIR = Path("/content/drive/MyDrive/Hockey Analysis Program")
OUTPUT_DIR = Path("/content/drive/MyDrive/HockeyLivePlayFrames")

MODEL_PATH = VIDEO_DIR / "ahl_shot_classifier_combined_1778167679.json"

VIDEO_GLOB = "HockeyUnedited*.mkv"
FRAME_EVERY_S = 1.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("model exists:", MODEL_PATH.exists())


In [ ]:
# ===== Cell 4 =====
!nvidia-smi

<details>
<summary>**Split the Broadcast into Camera-shot segments**</summary>

6. Split hockey broadcast into likely live-play segments and export sampled frames

</details>

In [ ]:
# ===== Cell 6 =====
import cv2
import json
import math
import pandas as pd
import numpy as np
from dataclasses import dataclass, asdict
from pathlib import Path
from tqdm.auto import tqdm


# Assumes these were already set in your path cell:
# VIDEO_DIR
# OUTPUT_DIR
# MODEL_PATH
# VIDEO_GLOB
# FRAME_EVERY_S


MIN_SEGMENT_S = 0.75
SCENE_SAMPLE_EVERY_S = 0.25
SCENE_CUT_THRESHOLD = 0.42
SCENE_HIST_BINS = (12, 6, 4)
SCENE_MIN_STRONG_CUT_DISTANCE = 0.60

CLOCK_BOX = (560, 65, 730, 120)

SEGMENT_START_BUFFER_S = 0.5
SEGMENT_END_BUFFER_S = 1.25

TRACKABLE_KEEP_THRESHOLD = 0.25

CLOCK_RUNNING_THRESHOLD = 4.0
CLOCK_STOPPED_THRESHOLD = 1.0
MODEL_STRONG_TRACKABLE_THRESHOLD = 0.75


@dataclass
class SegmentPrediction:
    video: str
    segment_index: int
    start_s: float
    end_s: float
    duration_s: float
    predicted_label: str
    predicted_confidence: float
    trackable_score: float
    non_trackable_score: float
    replay_score: float
    cut_score: float
    ice_ratio: float
    center_ice_ratio: float
    large_component_ratio: float
    edge_density: float
    mean_brightness: float
    mean_saturation: float
    clock_motion_score: float
    override_reason: str


def open_video(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    duration_s = frame_count / fps if frame_count else 0
    return cap, fps, duration_s


def read_frame_at(cap, fps: float, timestamp_s: float):
    frame_idx = int(round(timestamp_s * fps))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return frame


def frame_scene_signature(frame):
    small = cv2.resize(frame, (320, 180), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist(
        [hsv],
        [0, 1, 2],
        None,
        list(SCENE_HIST_BINS),
        [0, 180, 0, 256, 0, 256],
    )

    cv2.normalize(hist, hist, alpha=1.0, beta=0.0, norm_type=cv2.NORM_L1)
    return hist.flatten().astype(np.float32)


def histogram_cut_score(prev_hist, curr_hist):
    return float(cv2.compareHist(prev_hist, curr_hist, cv2.HISTCMP_BHATTACHARYYA))


def detect_shot_segments(video_path: Path):
    cap, fps, duration_s = open_video(video_path)

    segments = []
    prev_hist = None
    start_s = 0.0
    last_cut_score = 0.0

    sample_step_frames = max(1, int(round(SCENE_SAMPLE_EVERY_S * fps)))
    frame_index = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_index % sample_step_frames != 0:
            frame_index += 1
            continue

        t = frame_index / fps
        hist = frame_scene_signature(frame)

        if prev_hist is not None:
            score = histogram_cut_score(prev_hist, hist)
            elapsed = float(t) - start_s

            is_cut = score >= SCENE_CUT_THRESHOLD
            is_strong_cut = score >= SCENE_MIN_STRONG_CUT_DISTANCE

            if (is_cut and elapsed >= MIN_SEGMENT_S) or (is_strong_cut and elapsed >= 0.35):
                segments.append((start_s, float(t), last_cut_score))
                start_s = float(t)

            last_cut_score = score

        prev_hist = hist
        frame_index += 1

    if duration_s - start_s >= MIN_SEGMENT_S:
        segments.append((start_s, duration_s, last_cut_score))

    cap.release()
    return segments


def frame_features(frame):
    small = cv2.resize(frame, (640, 360), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

    h, saturation, value = cv2.split(hsv)

    ice_mask = ((value > 120) & (saturation < 95)).astype(np.uint8)
    ice_ratio = float(ice_mask.mean())

    height, width = ice_mask.shape
    center = ice_mask[int(height * 0.20):int(height * 0.85), int(width * 0.12):int(width * 0.88)]
    center_ice_ratio = float(center.mean()) if center.size else 0.0

    non_ice = (1 - ice_mask).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(non_ice, connectivity=8)

    largest = 0
    total_pixels = height * width
    for label in range(1, num_labels):
        area = int(stats[label, cv2.CC_STAT_AREA])
        largest = max(largest, area)

    large_component_ratio = largest / total_pixels if total_pixels else 0.0

    edges = cv2.Canny(gray, 60, 160)
    edge_density = float((edges > 0).mean())

    return {
        "ice_ratio": ice_ratio,
        "center_ice_ratio": center_ice_ratio,
        "large_component_ratio": large_component_ratio,
        "edge_density": edge_density,
        "mean_brightness": float(value.mean() / 255.0),
        "mean_saturation": float(saturation.mean() / 255.0),
    }


def clock_crop_gray_from_frame(frame, clock_box):
    x1, y1, x2, y2 = clock_box
    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return None

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (160, 50), interpolation=cv2.INTER_AREA)
    return gray


def median_clock_motion(samples):
    if len(samples) < 2:
        return 0.0

    diffs = []
    for prev, curr in zip(samples, samples[1:]):
        diffs.append(float(cv2.absdiff(prev, curr).mean()))

    return float(np.median(diffs)) if diffs else 0.0


def segment_features(video_path: Path, start_s: float, end_s: float):
    cap, fps, _ = open_video(video_path)

    duration_s = max(0.0, end_s - start_s)
    sample_times = np.linspace(start_s, end_s, num=min(5, max(2, int(duration_s) + 1)), endpoint=False)

    feature_rows = []
    clock_samples = []

    for timestamp in sample_times:
        frame = read_frame_at(cap, fps, float(timestamp))
        if frame is None:
            continue

        feature_rows.append(frame_features(frame))

        clock = clock_crop_gray_from_frame(frame, CLOCK_BOX)
        if clock is not None:
            clock_samples.append(clock)

    cap.release()

    if not feature_rows:
        return {
            "ice_ratio": 0.0,
            "center_ice_ratio": 0.0,
            "large_component_ratio": 1.0,
            "edge_density": 1.0,
            "mean_brightness": 0.0,
            "mean_saturation": 0.0,
            "clock_motion_score": 0.0,
        }

    med = {
        key: float(np.median([item[key] for item in feature_rows]))
        for key in feature_rows[0].keys()
    }

    med["clock_motion_score"] = median_clock_motion(clock_samples)
    return med


def load_model(model_path: Path):
    with model_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def gaussian_log_prob(x, mean, var):
    x = np.asarray(x, dtype=float)
    mean = np.asarray(mean, dtype=float)
    var = np.asarray(var, dtype=float)
    return float(-0.5 * np.sum(np.log(2 * np.pi * var) + ((x - mean) ** 2) / var))


def predict_segment_label(model, feature_values):
    scores = {}

    for cls in model["classes"]:
        label = cls["label"]
        prior = max(float(cls["prior"]), 1e-9)
        score = math.log(prior) + gaussian_log_prob(feature_values, cls["mean"], cls["variance"])
        scores[label] = score

    max_score = max(scores.values())
    exp_scores = {k: math.exp(v - max_score) for k, v in scores.items()}
    total = sum(exp_scores.values()) or 1.0
    probs = {k: v / total for k, v in exp_scores.items()}

    label = max(probs, key=probs.get)
    return label, float(probs[label]), probs


def apply_clock_override(label, confidence, probs, features):
    clock_score = float(features.get("clock_motion_score", 0.0))
    trackable_score = float(probs.get("trackable_rink", 0.0))

    override_reason = ""

    if clock_score >= CLOCK_RUNNING_THRESHOLD:
        label = "trackable_rink"
        confidence = max(confidence, trackable_score, 0.99)
        probs["trackable_rink"] = max(probs.get("trackable_rink", 0.0), 0.99)
        probs["non_trackable"] = min(probs.get("non_trackable", 0.0), 0.01)
        override_reason = "clock_running_override"

    elif clock_score <= CLOCK_STOPPED_THRESHOLD and trackable_score < MODEL_STRONG_TRACKABLE_THRESHOLD:
        label = "non_trackable"
        confidence = max(confidence, probs.get("non_trackable", 0.0), 0.99)
        probs["non_trackable"] = max(probs.get("non_trackable", 0.0), 0.99)
        probs["trackable_rink"] = min(probs.get("trackable_rink", 0.0), 0.01)
        override_reason = "clock_stopped_override"

    return label, confidence, probs, override_reason


def should_export_segment(row):
    return (
        row.predicted_label == "trackable_rink"
        or row.trackable_score >= TRACKABLE_KEEP_THRESHOLD
    )


def classify_video_segments(video_path: Path, model):
    raw_segments = detect_shot_segments(video_path)
    rows = []

    feature_names = model["feature_names"]

    for idx, (start_s, end_s, cut_score) in enumerate(tqdm(raw_segments, desc=f"Classifying {video_path.name}")):
        duration_s = end_s - start_s
        features = segment_features(video_path, start_s, end_s)

        feature_map = {
            "duration_s": duration_s,
            "cut_score": cut_score,
            **features,
        }

        feature_values = [float(feature_map.get(name, 0.0)) for name in feature_names]
        label, confidence, probs = predict_segment_label(model, feature_values)

        label, confidence, probs, override_reason = apply_clock_override(
            label,
            confidence,
            probs,
            features,
        )

        rows.append(
            SegmentPrediction(
                video=video_path.name,
                segment_index=idx,
                start_s=round(float(start_s), 3),
                end_s=round(float(end_s), 3),
                duration_s=round(float(duration_s), 3),
                predicted_label=label,
                predicted_confidence=round(confidence, 6),
                trackable_score=round(float(probs.get("trackable_rink", 0.0)), 6),
                non_trackable_score=round(float(probs.get("non_trackable", 0.0)), 6),
                replay_score=round(float(probs.get("replay", 0.0)), 6),
                cut_score=round(float(cut_score), 6),
                ice_ratio=round(float(features["ice_ratio"]), 6),
                center_ice_ratio=round(float(features["center_ice_ratio"]), 6),
                large_component_ratio=round(float(features["large_component_ratio"]), 6),
                edge_density=round(float(features["edge_density"]), 6),
                mean_brightness=round(float(features["mean_brightness"]), 6),
                mean_saturation=round(float(features["mean_saturation"]), 6),
                clock_motion_score=round(float(features["clock_motion_score"]), 6),
                override_reason=override_reason,
            )
        )

    return rows


def export_frames_from_predictions(video_path: Path, predictions, output_dir: Path, every_s: float = 1.0):
    cap, fps, _ = open_video(video_path)

    video_out = output_dir / video_path.stem
    frames_out = video_out / "frames"
    frames_out.mkdir(parents=True, exist_ok=True)

    exported = []

    kept_segments = [
        row for row in predictions
        if should_export_segment(row)
    ]

    for seg in tqdm(kept_segments, desc=f"Exporting {video_path.name}"):
        t = seg.start_s + SEGMENT_START_BUFFER_S
        stop_t = seg.end_s - SEGMENT_END_BUFFER_S

        while t < stop_t:
            frame = read_frame_at(cap, fps, t)

            if frame is not None:
                frame_idx = int(round(t * fps))
                name = f"{video_path.stem}_seg{seg.segment_index:04d}_t{t:08.2f}_f{frame_idx:08d}.jpg"
                path = frames_out / name

                cv2.imwrite(str(path), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])

                exported.append({
                    "video": video_path.name,
                    "segment_index": seg.segment_index,
                    "timestamp_s": round(float(t), 3),
                    "frame_idx": frame_idx,
                    "predicted_label": seg.predicted_label,
                    "predicted_confidence": seg.predicted_confidence,
                    "trackable_score": seg.trackable_score,
                    "clock_motion_score": seg.clock_motion_score,
                    "override_reason": seg.override_reason,
                    "exported_by": "label" if seg.predicted_label == "trackable_rink" else "trackable_score_threshold",
                    "image_path": str(path),
                })

            t += every_s

    cap.release()

    pd.DataFrame(exported).to_csv(video_out / "exported_frames.csv", index=False)
    return exported


def process_video_with_model(video_path: Path, model, output_dir: Path):
    video_out = output_dir / video_path.stem
    video_out.mkdir(parents=True, exist_ok=True)

    predictions = classify_video_segments(video_path, model)

    predictions_csv = video_out / "segment_predictions.csv"
    pd.DataFrame([asdict(row) for row in predictions]).to_csv(predictions_csv, index=False)

    exported = export_frames_from_predictions(video_path, predictions, output_dir, every_s=FRAME_EVERY_S)

    print()
    print(video_path.name)
    print("segments:", len(predictions))
    print("predicted trackable:", sum(1 for row in predictions if row.predicted_label == "trackable_rink"))
    print("predicted non_trackable:", sum(1 for row in predictions if row.predicted_label == "non_trackable"))
    print("predicted replay:", sum(1 for row in predictions if row.predicted_label == "replay"))
    print("clock running overrides:", sum(1 for row in predictions if row.override_reason == "clock_running_override"))
    print("clock stopped overrides:", sum(1 for row in predictions if row.override_reason == "clock_stopped_override"))
    print("kept by export rule:", sum(1 for row in predictions if should_export_segment(row)))
    print("exported frames:", len(exported))
    print("predictions csv:", predictions_csv)
    print("frames csv:", video_out / "exported_frames.csv")

    return predictions, exported


def write_segment_cut_review(video_path: Path, predictions, output_dir: Path, max_segments=None, edge_padding_s=0.25):
    review_dir = output_dir / video_path.stem / "cut_review"
    review_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)

    rows = []
    selected = predictions if max_segments is None else predictions[:max_segments]

    for seg in selected:
        safe_start = min(seg.end_s, seg.start_s + edge_padding_s)
        safe_end = max(seg.start_s, seg.end_s - edge_padding_s)

        times = [
            ("start+pad", safe_start),
            ("middle", (seg.start_s + seg.end_s) / 2),
            ("end-pad", safe_end),
        ]

        frames = []

        for label_text, t in times:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(t * fps))
            ok, frame = cap.read()

            if not ok:
                frame = np.zeros((270, 480, 3), dtype=np.uint8)
            else:
                frame = cv2.resize(frame, (480, 270), interpolation=cv2.INTER_AREA)

            text = f"{label_text}: {t:.2f}s"
            cv2.putText(frame, text, (14, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 255), 2)
            frames.append(frame)

        sheet = np.concatenate(frames, axis=1)

        name = (
            f"segment_{seg.segment_index:04d}_"
            f"{seg.start_s:08.2f}_{seg.end_s:08.2f}_"
            f"{seg.predicted_label}_"
            f"score{seg.predicted_confidence:.2f}_"
            f"trackable{seg.trackable_score:.2f}_"
            f"clock{seg.clock_motion_score:.2f}_"
            f"{seg.override_reason or 'no_override'}.jpg"
        )

        path = review_dir / name
        cv2.imwrite(str(path), sheet, [cv2.IMWRITE_JPEG_QUALITY, 92])

        rows.append({
            "segment_index": seg.segment_index,
            "start_s": seg.start_s,
            "end_s": seg.end_s,
            "duration_s": seg.duration_s,
            "predicted_label": seg.predicted_label,
            "predicted_confidence": seg.predicted_confidence,
            "trackable_score": seg.trackable_score,
            "non_trackable_score": seg.non_trackable_score,
            "replay_score": seg.replay_score,
            "clock_motion_score": seg.clock_motion_score,
            "override_reason": seg.override_reason,
            "exported": should_export_segment(seg),
            "cut_score": seg.cut_score,
            "review_image": str(path),
        })

    cap.release()

    review_csv = review_dir / "cut_review.csv"
    pd.DataFrame(rows).to_csv(review_csv, index=False)

    print("review_dir:", review_dir)
    print("review_csv:", review_csv)
    print("review images:", len(rows))

    return review_dir, review_csv


model = load_model(MODEL_PATH)
print("loaded model:", MODEL_PATH)
print("model classes:", [(c["label"], c["count"]) for c in model["classes"]])

videos_to_process = [
    "HockeyUnedited1.mkv",
    "HockeyUnedited2.mkv",
    "HockeyUnedited3.mkv",
    "HockeyUnedited4.mkv",
]

all_predictions = []
all_exported = []

for video_name in videos_to_process:
    video_path = VIDEO_DIR / video_name

    if not video_path.exists():
        print("SKIPPING missing video:", video_path)
        continue

    predictions, exported = process_video_with_model(video_path, model, OUTPUT_DIR)
    write_segment_cut_review(video_path, predictions, OUTPUT_DIR)

    all_predictions.extend(predictions)
    all_exported.extend(exported)

print()
print("DONE")
print("processed videos:", videos_to_process)
print("total segment predictions:", len(all_predictions))
print("total exported frames:", len(all_exported))







<details>
<summary>RF-DETR Small Training and Inference</summary>

8. Set RF-DETR dataset and output paths
9. Install RF-DETR and Supervision
10. Unzip RF-DETR COCO dataset
11. Find the COCO dataset root folder
12. Check class names and annotation counts
13. Clear old model/GPU memory/output folder
14. Train RF-DETR Small detector
15. Save RF-DETR training output to Google Drive
16. Clear detector from memory
17. Load best RF-DETR checkpoint for inference


</details>

In [ ]:
# ===== Cell 8 =====
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "Hockey Analysis Program"

DATASET_ZIP = PROJECT_DIR / "HockeyAnalysisProgram.v1i.coco.zip"
DATASET_DIR = Path("/content/hockey_detector_dataset")
DATASET_DETECTION_ROOT = Path("/content/hockey_detector_dataset_bbox")

OUTPUT_DRIVE_DIR = PROJECT_DIR / "rf_detr_hockey_small_output"

print("zip exists:", DATASET_ZIP.exists())
print("zip path:", DATASET_ZIP)

In [ ]:
# ===== Cell 9 =====
!pip install -q "rfdetr>=1.4.0" supervision
!pip install -q "rfdetr[train,loggers]>=1.4.0" supervision


In [ ]:
# ===== Cell 10 =====
import shutil

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

DATASET_DIR.mkdir(parents=True, exist_ok=True)

!unzip -q "{DATASET_ZIP}" -d "{DATASET_DIR}"

print("unzipped to:", DATASET_DIR)
!find "{DATASET_DIR}" -maxdepth 3 -type f | head -50

In [ ]:
# ===== Cell 11 =====
def find_coco_dataset_root(root: Path):
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for candidate in candidates:
        train_ann = candidate / "train" / "_annotations.coco.json"
        valid_ann = candidate / "valid" / "_annotations.coco.json"

        if train_ann.exists() and valid_ann.exists():
            return candidate

    raise FileNotFoundError("Could not find train/valid COCO annotations.")

DATASET_ROOT = find_coco_dataset_root(DATASET_DIR)

print("DATASET_ROOT:", DATASET_ROOT)
print("train:", (DATASET_ROOT / "train" / "_annotations.coco.json").exists())
print("valid:", (DATASET_ROOT / "valid" / "_annotations.coco.json").exists())
print("test:", (DATASET_ROOT / "test" / "_annotations.coco.json").exists())


In [ ]:
# ===== Cell 12 =====
import json
from collections import Counter

ann_path = DATASET_ROOT / "train" / "_annotations.coco.json"
data = json.loads(ann_path.read_text())

categories = {cat["id"]: cat["name"] for cat in data["categories"]}
counts = Counter(categories[ann["category_id"]] for ann in data["annotations"])

print("classes:")
for category_id, name in categories.items():
    print(category_id, name)

print()
print("train annotation counts:")
for name, count in counts.most_common():
    print(name, count)


In [ ]:
# ===== Cell 13 =====
import gc
import shutil
import torch
from pathlib import Path

try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

if Path("/content/output").exists():
    shutil.rmtree("/content/output")

print("cleaned")

In [ ]:
# ===== Cell 14 =====
from rfdetr import RFDETRSmall

model = RFDETRSmall(resolution=512)

model.train(
    dataset_dir=str(DATASET_ROOT),
    epochs=25,
    batch_size=4,
    grad_accum_steps=4,
    resolution=512,
)


In [ ]:
# ===== Cell 15 =====
import shutil
from pathlib import Path

OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for item in Path("/content/output").iterdir():
    dst = OUTPUT_DRIVE_DIR / item.name

    if item.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(item, dst)
    else:
        shutil.copy2(item, dst)

print("saved to:", OUTPUT_DRIVE_DIR)
!ls -lah "{OUTPUT_DRIVE_DIR}"


In [ ]:
# ===== Cell 16 =====
import gc
import torch

try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ===== Cell 17 =====
from rfdetr import RFDETRSmall

BEST_CHECKPOINT = OUTPUT_DRIVE_DIR / "checkpoint_best_total.pth"

model = RFDETRSmall(pretrain_weights=str(BEST_CHECKPOINT), resolution=512)
model.optimize_for_inference()

print("loaded:", BEST_CHECKPOINT)


<details>
<summary>Detector Inference</summary>

19. Set detector inference input/output paths
20. Load trained RF-DETR detector
21. Define detector class-name mapping
22. Load exported live-play frame CSVs
23. Preview detector results on random frames
24. Run detector on all live-play frames and save detections/annotations
25. Display grid preview of annotated detector frames


</details>

In [ ]:
# ===== Cell 19 =====
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "Hockey Analysis Program"

TRACKABLE_FRAMES_ROOT = DRIVE_ROOT / "HockeyLivePlayFrames"
CHECKPOINT_PATH = PROJECT_DIR / "rf_detr_hockey_small_output" / "checkpoint_best_total.pth"

INFERENCE_OUT_DIR = PROJECT_DIR / "rf_detr_hockey_small_inference_review"
INFERENCE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("checkpoint exists:", CHECKPOINT_PATH.exists())
print("trackable frames root exists:", TRACKABLE_FRAMES_ROOT.exists())
print("output:", INFERENCE_OUT_DIR)


In [ ]:
# ===== Cell 20 =====
from rfdetr import RFDETRSmall

model = RFDETRSmall(pretrain_weights=str(CHECKPOINT_PATH), resolution=512)
model.optimize_for_inference()

print("loaded detector")


In [ ]:
# ===== Cell 21 =====
CLASS_ID_TO_NAME = {
    1: "Player Jersey number",
    2: "Player Skating",
    3: "Referee",
}

def class_name_from_id(class_id):
    class_id = int(class_id)

    if class_id in CLASS_ID_TO_NAME:
        return CLASS_ID_TO_NAME[class_id]

    zero_based = class_id + 1
    if zero_based in CLASS_ID_TO_NAME:
        return CLASS_ID_TO_NAME[zero_based]

    return f"class_{class_id}"



In [ ]:
# ===== Cell 22 =====
import pandas as pd

all_rows = []

for csv_path in sorted(TRACKABLE_FRAMES_ROOT.glob("HockeyUnedited*/exported_frames.csv")):
    df = pd.read_csv(csv_path)

    if len(df) == 0:
        continue

    df["source_csv"] = str(csv_path)
    df["source_video_folder"] = csv_path.parent.name
    all_rows.append(df)

if not all_rows:
    raise RuntimeError("No exported_frames.csv files found.")

frames_df = pd.concat(all_rows, ignore_index=True)

frames_df["image_path"] = frames_df["image_path"].astype(str)
frames_df = frames_df[frames_df["image_path"].apply(lambda p: Path(p).exists())].copy()

print("trackable frames found:", len(frames_df))
print(frames_df["source_video_folder"].value_counts())
display(frames_df.head())


In [ ]:
# ===== Cell 23 =====
import random
import math
import supervision as sv
from PIL import Image
from IPython.display import display

THRESHOLD = 0.35
PREVIEW_COUNT = min(12, len(frames_df))

sample_df = frames_df.sample(n=PREVIEW_COUNT, random_state=42).reset_index(drop=True)

thumb_w, thumb_h = 480, 270
cols = 3
rows = math.ceil(PREVIEW_COUNT / cols)

sheet = Image.new("RGB", (cols * thumb_w, rows * thumb_h), "white")

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.45, text_thickness=1)

for i, row in sample_df.iterrows():
    image = Image.open(row["image_path"]).convert("RGB")
    detections = model.predict(image, threshold=THRESHOLD)

    labels = [
        f"{class_name_from_id(class_id)} {confidence:.2f}"
        for class_id, confidence
        in zip(detections.class_id, detections.confidence)
    ]

    annotated = image.copy()
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    annotated = annotated.resize((thumb_w, thumb_h))

    x = (i % cols) * thumb_w
    y = (i // cols) * thumb_h
    sheet.paste(annotated, (x, y))

display(sheet)


In [ ]:
# ===== Cell 24 =====
from tqdm.auto import tqdm
import supervision as sv
from PIL import Image
import pandas as pd
import shutil
from pathlib import Path

THRESHOLD = 0.35
MAX_ANNOTATED_IMAGES = 500

CLASS_ID_TO_NAME = {
    1: "Player Jersey number",
    2: "Player Skating",
    3: "Referee",
}

def class_name_from_id(class_id):
    class_id = int(class_id)

    if class_id in CLASS_ID_TO_NAME:
        return CLASS_ID_TO_NAME[class_id]

    return f"class_{class_id}"


annotated_dir = INFERENCE_OUT_DIR / "annotated_frames"

if annotated_dir.exists():
    shutil.rmtree(annotated_dir)

annotated_dir.mkdir(parents=True, exist_ok=True)

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.45, text_thickness=1)

detection_rows = []
image_summary_rows = []

for idx, row in tqdm(frames_df.reset_index(drop=True).iterrows(), total=len(frames_df)):
    image_path = Path(row["image_path"])
    image = Image.open(image_path).convert("RGB")

    detections = model.predict(image, threshold=THRESHOLD)

    labels = [
        f"{class_name_from_id(class_id)} {confidence:.2f}"
        for class_id, confidence
        in zip(detections.class_id, detections.confidence)
    ]

    image_summary_rows.append({
        "image_path": str(image_path),
        "source_video": row.get("video", ""),
        "source_video_folder": row.get("source_video_folder", ""),
        "segment_index": row.get("segment_index", ""),
        "timestamp_s": row.get("timestamp_s", ""),
        "num_detections": len(detections),
    })

    for det_i, (xyxy, class_id, confidence) in enumerate(
        zip(detections.xyxy, detections.class_id, detections.confidence)
    ):
        x1, y1, x2, y2 = [float(v) for v in xyxy]

        detection_rows.append({
            "image_path": str(image_path),
            "source_video": row.get("video", ""),
            "source_video_folder": row.get("source_video_folder", ""),
            "segment_index": row.get("segment_index", ""),
            "timestamp_s": row.get("timestamp_s", ""),
            "detection_index": det_i,
            "class_id": int(class_id),
            "class_name": class_name_from_id(class_id),
            "confidence": float(confidence),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

    if idx < MAX_ANNOTATED_IMAGES:
        annotated = image.copy()
        annotated = box_annotator.annotate(annotated, detections)
        annotated = label_annotator.annotate(annotated, detections, labels)

        out_name = f"{idx:05d}_{image_path.name}"
        annotated.save(annotated_dir / out_name, quality=92)

detections_df = pd.DataFrame(detection_rows)
summary_df = pd.DataFrame(image_summary_rows)

detections_csv = INFERENCE_OUT_DIR / "detections.csv"
summary_csv = INFERENCE_OUT_DIR / "image_summary.csv"

detections_df.to_csv(detections_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print("images processed:", len(frames_df))
print("detections:", len(detections_df))
print("detections csv:", detections_csv)
print("summary csv:", summary_csv)
print("annotated images:", annotated_dir)
display(detections_df.head())


In [ ]:
# ===== Cell 25 =====
import math
from PIL import Image
from IPython.display import display

annotated_paths = sorted(annotated_dir.glob("*.jpg"))

PREVIEW_COUNT = min(60, len(annotated_paths))
thumb_w, thumb_h = 320, 180
cols = 4
rows = math.ceil(PREVIEW_COUNT / cols)

sheet = Image.new("RGB", (cols * thumb_w, rows * thumb_h), "white")

for i, path in enumerate(annotated_paths[:PREVIEW_COUNT]):
    img = Image.open(path).convert("RGB").resize((thumb_w, thumb_h))

    x = (i % cols) * thumb_w
    y = (i // cols) * thumb_h
    sheet.paste(img, (x, y))

print("showing:", PREVIEW_COUNT)
display(sheet)


<details>
<summary>SAM2 Integration</summary>

27. Set SAM2 output paths and choose source video
28. Clone SAM2 real-time repository
29. Install SAM2 real-time dependencies
30. Download SAM2 checkpoint
31. Set SAM2 checkpoint/config paths
32. Load RF-DETR detector for SAM2 prompting
33. Define detector class names and player class IDs
34. Load existing detector detections CSV
35. Compress/copy SAM2 source clip
36. Define player detection helper for SAM2 prompt frames
37. Define SAM2 tracking helper functions
38. Load SAM2 real-time predictor
39. Define SAM2 chunked tracking and re-identification helpers
40. Run chunked SAM2 tracking and save masks/tracks/prompts
41. Compress and preview SAM2 tracking output video

</details>

In [ ]:
# ===== Cell 27 =====
from pathlib import Path
import shutil

HOME = Path("/content")
SAM2_DIR = HOME / "segment-anything-2-real-time"

print("HOME:", HOME)

if SAM2_DIR.exists():
    shutil.rmtree(SAM2_DIR)


In [ ]:
# ===== Cell 28 =====
%cd /content

!git clone https://github.com/Gy920/segment-anything-2-real-time.git

%cd /content/segment-anything-2-real-time


!pip install -q hydra-core omegaconf iopath ninja matplotlib

!pip install -e . --no-build-isolation -q || true

!python setup.py build_ext --inplace



In [ ]:
# ===== Cell 29 =====
%cd /content/segment-anything-2-real-time

import sys
sys.path.insert(0, "/content/segment-anything-2-real-time")

from sam2.build_sam import build_sam2_camera_predictor

print("SAM2 import worked")


In [ ]:
# ===== Cell 30 =====
%cd /content/segment-anything-2-real-time

!(cd checkpoints && bash download_ckpts.sh)

from pathlib import Path

print("large checkpoint exists:", Path("checkpoints/sam2.1_hiera_large.pt").exists())



In [ ]:
# ===== Cell 31 =====
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "Hockey Analysis Program"
LIVE_FRAMES_ROOT = DRIVE_ROOT / "HockeyLivePlayFrames"

VIDEO_DIR = PROJECT_DIR
CHECKPOINT_PATH = PROJECT_DIR / "rf_detr_hockey_small_output" / "checkpoint_best_total.pth"

SAM2_OUT_DIR = PROJECT_DIR / "sam2_hockey_tracking_review"
SAM2_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("detector checkpoint exists:", CHECKPOINT_PATH.exists())
print("output:", SAM2_OUT_DIR)



In [ ]:
# ===== Cell 32 =====
from rfdetr import RFDETRSmall

PLAYER_DETECTION_MODEL_CONFIDENCE = 0.35

detector = RFDETRSmall(pretrain_weights=str(CHECKPOINT_PATH), resolution=512)
detector.optimize_for_inference()

print("loaded RF-DETR detector")



In [ ]:
# ===== Cell 33 =====
CLASS_ID_TO_NAME = {
    1: "Player Jersey number",
    2: "Player Skating",
    3: "Referee",
}

PLAYER_CLASS_IDS = [2]


In [ ]:
# ===== Cell 34 =====
import pandas as pd

VIDEO_NAME = "HockeyUnedited4"

SOURCE_FULL_VIDEO = VIDEO_DIR / f"{VIDEO_NAME}.mkv"
PREDICTIONS_CSV = LIVE_FRAMES_ROOT / VIDEO_NAME / "segment_predictions.csv"

pred_df = pd.read_csv(PREDICTIONS_CSV)

trackable_df = pred_df[
    (pred_df["predicted_label"] == "trackable_rink") |
    (pred_df["override_reason"] == "clock_running_override")
].copy()

trackable_df = trackable_df.sort_values("duration_s", ascending=False).reset_index(drop=True)

print("trackable segments:", len(trackable_df))
display(trackable_df[[
    "segment_index",
    "start_s",
    "end_s",
    "duration_s",
    "predicted_label",
    "trackable_score",
    "clock_motion_score",
    "override_reason",
]].head(10))


In [ ]:
# ===== Cell 35 =====

from pathlib import Path

SAM2_OUT_DIR = Path("/content/drive/MyDrive/Hockey Analysis Program/sam2_hockey_tracking_review")
SAM2_OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_VIDEO_PATH = SAM2_OUT_DIR / "HockeyUnedited4_seg0021_sam2_test.mp4"

print("SAM2 source video:")
print(SOURCE_VIDEO_PATH)
print("Exists:", SOURCE_VIDEO_PATH.exists())

if not SOURCE_VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Could not find SOURCE_VIDEO_PATH:\n{SOURCE_VIDEO_PATH}\n\n"
        "Make sure the demo clip exists in the sam2_hockey_tracking_review folder."
    )

# Output paths for SAM2 tracking
TARGET_VIDEO_PATH = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_masks.mp4"
TARGET_VIDEO_COMPRESSED_PATH = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_masks_compressed.mp4"
TRACKS_CSV = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_tracks.csv"
PROMPTS_CSV = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_prompts.csv"

print("\nSAM2 output paths:")
print("Raw mask video:", TARGET_VIDEO_PATH)
print("Compressed mask video:", TARGET_VIDEO_COMPRESSED_PATH)
print("Tracks CSV:", TRACKS_CSV)
print("Prompts CSV:", PROMPTS_CSV)


In [ ]:
# ===== Cell 36 =====
import cv2
import numpy as np
import supervision as sv
from PIL import Image
from IPython.display import display

def detect_players_in_frame(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    detections = detector.predict(image, threshold=PLAYER_DETECTION_MODEL_CONFIDENCE)

    detections = detections[np.isin(detections.class_id, PLAYER_CLASS_IDS)]
    return detections


frame_generator = sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH))
first_frame = next(frame_generator)

detections = detect_players_in_frame(first_frame)
detections.tracker_id = np.arange(1, len(detections) + 1)

print("player detections:", len(detections))
print("class ids:", detections.class_id)

box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1, color_lookup=sv.ColorLookup.TRACK)

labels = [f"player {tid}" for tid in detections.tracker_id]

annotated = first_frame.copy()
annotated = box_annotator.annotate(annotated, detections)
annotated = label_annotator.annotate(annotated, detections, labels)

display(Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)).resize((960, 540)))


In [ ]:
# ===== Cell 37 =====
import cv2
import numpy as np

def clean_mask_by_centroid_distance(
    mask,
    min_component_area=80,
    max_centroid_distance_ratio=0.06,
):
    mask_u8 = mask.astype(np.uint8)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)

    if num_labels <= 1:
        return mask.astype(bool)

    areas = stats[1:, cv2.CC_STAT_AREA]
    if len(areas) == 0:
        return mask.astype(bool)

    largest_label = int(np.argmax(areas)) + 1
    main_centroid = centroids[largest_label]

    h, w = mask.shape
    max_dist = max_centroid_distance_ratio * float(np.hypot(h, w))

    cleaned = labels == largest_label

    for label_id in range(1, num_labels):
        if label_id == largest_label:
            continue

        area = int(stats[label_id, cv2.CC_STAT_AREA])
        if area < min_component_area:
            continue

        dist = float(np.linalg.norm(centroids[label_id] - main_centroid))

        if dist <= max_dist:
            cleaned |= labels == label_id

    return cleaned.astype(bool)


In [ ]:
# ===== Cell 38 =====
%cd /content/segment-anything-2-real-time

import sys
import torch

sys.path.insert(0, "/content/segment-anything-2-real-time")

from sam2.build_sam import build_sam2_camera_predictor

SAM2_CHECKPOINT = "checkpoints/sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"

predictor = build_sam2_camera_predictor(SAM2_CONFIG, SAM2_CHECKPOINT)

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("loaded fresh SAM2 large")
print("AMP dtype:", AMP_DTYPE)



In [ ]:
# ===== Cell 39 =====

import numpy as np
import cv2
import supervision as sv


CHUNK_RESTART_EVERY_S = 3


def box_iou(box_a, box_b):
    """
    Compute IoU between two boxes: [x1, y1, x2, y2]
    """
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)

    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0

    return inter_area / union


def appearance_embedding(frame, xyxy):
    """
    Simple color histogram embedding for re-identifying players across SAM2 chunks.
    """
    x1, y1, x2, y2 = [int(v) for v in xyxy]

    h, w = frame.shape[:2]

    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h, y2))

    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return np.zeros(48, dtype=np.float32)

    crop_hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist(
        [crop_hsv],
        [0, 1, 2],
        None,
        [8, 3, 2],
        [0, 180, 0, 256, 0, 256]
    )

    hist = hist.flatten().astype(np.float32)

    norm = np.linalg.norm(hist)

    if norm > 0:
        hist = hist / norm

    return hist


def cosine_similarity(a, b):
    """
    Cosine similarity between two vectors.
    """
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom <= 0:
        return 0.0

    return float(np.dot(a, b) / denom)


def assign_stitched_tracker_ids(
    detections,
    previous_track_state,
    frame,
    next_tracker_id,
    iou_weight=0.7,
    appearance_weight=0.3,
    match_threshold=0.15,
):
    """
    Assign stable tracker IDs at each SAM2 re-prompt frame.

    Uses box IoU + simple color appearance similarity to connect new RF-DETR detections
    with previous SAM2 tracks.
    """
    if len(detections) == 0:
        detections.tracker_id = np.array([], dtype=int)
        return detections, next_tracker_id

    assigned_ids = []
    used_previous_ids = set()

    for xyxy in detections.xyxy:
        current_appearance = appearance_embedding(frame, xyxy)

        best_id = None
        best_score = -1.0

        for previous_id, state in previous_track_state.items():
            if previous_id in used_previous_ids:
                continue

            previous_box = state["xyxy"]
            previous_appearance = state["appearance"]

            iou = box_iou(xyxy, previous_box)
            app_sim = cosine_similarity(current_appearance, previous_appearance)

            score = (iou_weight * iou) + (appearance_weight * app_sim)

            if score > best_score:
                best_score = score
                best_id = previous_id

        if best_id is not None and best_score >= match_threshold:
            assigned_ids.append(int(best_id))
            used_previous_ids.add(int(best_id))
        else:
            assigned_ids.append(int(next_tracker_id))
            next_tracker_id += 1

    detections.tracker_id = np.asarray(assigned_ids, dtype=int)

    return detections, next_tracker_id


class ChunkedSAM2Tracker:
    """
    Small wrapper around the SAM2 real-time camera predictor.

    It lets us:
    1. Re-prompt SAM2 with RF-DETR boxes at chunk boundaries.
    2. Propagate masks between prompt frames.
    3. Return supervision.Detections with xyxy, masks, and tracker_id.
    """

    def __init__(self, predictor):
        self.predictor = predictor
        self.tracker_ids = []

    def prompt_frame(self, frame, detections):
        """
        Reset/reload SAM2 on a prompt frame and add all player boxes.
        """
        self.tracker_ids = []

        self.predictor.load_first_frame(frame)

        if len(detections) == 0:
            return

        for xyxy, tracker_id in zip(detections.xyxy, detections.tracker_id):
            box = np.asarray(xyxy, dtype=np.float32)

            self.predictor.add_new_prompt(
                frame_idx=0,
                obj_id=int(tracker_id),
                bbox=box,
            )

            self.tracker_ids.append(int(tracker_id))

    def propagate(self, frame):
        """
        Track existing prompted objects into the next frame.
        """
        if len(self.tracker_ids) == 0:
            return sv.Detections.empty()

        out_obj_ids, out_mask_logits = self.predictor.track(frame)

        masks = []
        xyxy_list = []
        tracker_ids = []

        for obj_id, mask_logit in zip(out_obj_ids, out_mask_logits):
            mask = (mask_logit[0].detach().cpu().numpy() > 0.0)

            if mask.sum() == 0:
                continue

            ys, xs = np.where(mask)

            x1 = float(xs.min())
            y1 = float(ys.min())
            x2 = float(xs.max())
            y2 = float(ys.max())

            masks.append(mask)
            xyxy_list.append([x1, y1, x2, y2])
            tracker_ids.append(int(obj_id))

        if len(xyxy_list) == 0:
            return sv.Detections.empty()

        detections = sv.Detections(
            xyxy=np.asarray(xyxy_list, dtype=np.float32),
            mask=np.asarray(masks, dtype=bool),
            confidence=np.ones(len(xyxy_list), dtype=np.float32),
            class_id=np.zeros(len(xyxy_list), dtype=int),
            tracker_id=np.asarray(tracker_ids, dtype=int),
        )

        return detections


print("SAM2 helper definitions loaded.")
print("CHUNK_RESTART_EVERY_S:", CHUNK_RESTART_EVERY_S)

In [ ]:
# ===== Cell 40 =====
from tqdm.auto import tqdm
TARGET_VIDEO_PATH = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_masks.mp4"
TARGET_VIDEO_COMPRESSED_PATH = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_masks_compressed.mp4"
TRACKS_CSV = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_tracks.csv"
PROMPTS_CSV = SAM2_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_chunked_reid_sam2_prompts.csv"

CHUNK_RESTART_EVERY_S = 3

TRACK_COLORS = sv.ColorPalette.from_hex([
    "#00BFFF", "#FF4D4D", "#66FF66", "#FFD166", "#B967FF",
    "#FF8C42", "#3DDC97", "#F72585", "#4CC9F0", "#F9C74F",
    "#90BE6D", "#577590", "#43AA8B", "#F3722C", "#277DA1",
    "#E76F51", "#2A9D8F", "#E9C46A", "#264653", "#A7C957",
])

mask_annotator = sv.MaskAnnotator(
    color=TRACK_COLORS,
    color_lookup=sv.ColorLookup.TRACK,
    opacity=0.55,
)

box_annotator = sv.BoxAnnotator(
    color=TRACK_COLORS,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
)

label_annotator = sv.LabelAnnotator(
    color=TRACK_COLORS,
    color_lookup=sv.ColorLookup.TRACK,
    text_scale=0.5,
    text_thickness=1,
)

video_info = sv.VideoInfo.from_video_path(str(SOURCE_VIDEO_PATH))
restart_every_n_frames = max(1, int(round(video_info.fps * CHUNK_RESTART_EVERY_S)))

print("video fps:", video_info.fps)
print("restart every frames:", restart_every_n_frames)

frame_generator = sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH))

chunked_tracker = ChunkedSAM2Tracker(predictor)

next_tracker_id = 1
previous_track_state = {}

track_rows = []
prompt_rows = []

with sv.VideoSink(str(TARGET_VIDEO_PATH), video_info) as sink:
    for frame_idx, frame in tqdm(enumerate(frame_generator), total=video_info.total_frames):
        should_restart_chunk = frame_idx == 0 or frame_idx % restart_every_n_frames == 0

        if should_restart_chunk:
            player_detections = detect_players_in_frame(frame)

            player_detections, next_tracker_id = assign_stitched_tracker_ids(
                detections=player_detections,
                previous_track_state=previous_track_state,
                frame=frame,
                next_tracker_id=next_tracker_id,
            )

            chunked_tracker.prompt_frame(frame, player_detections)
            active_detections = player_detections

            previous_track_state = {
                int(tid): {
                    "xyxy": np.asarray(xyxy, dtype=np.float32),
                    "appearance": appearance_embedding(frame, xyxy),
                }
                for xyxy, tid in zip(active_detections.xyxy, active_detections.tracker_id)
            }

            for xyxy, tracker_id in zip(active_detections.xyxy, active_detections.tracker_id):
                prompt_rows.append({
                    "frame_idx": int(frame_idx),
                    "tracker_id": int(tracker_id),
                    "prompt_type": "chunk_prompt",
                    "x1": float(xyxy[0]),
                    "y1": float(xyxy[1]),
                    "x2": float(xyxy[2]),
                    "y2": float(xyxy[3]),
                })

        else:
            active_detections = chunked_tracker.propagate(frame)

            previous_track_state = {
                int(tid): {
                    "xyxy": np.asarray(xyxy, dtype=np.float32),
                    "appearance": appearance_embedding(frame, xyxy),
                }
                for xyxy, tid in zip(active_detections.xyxy, active_detections.tracker_id)
            }

        labels = [f"id {tid}" for tid in active_detections.tracker_id]

        annotated = frame.copy()

        if active_detections.mask is not None and len(active_detections.mask) > 0:
            annotated = mask_annotator.annotate(annotated, active_detections)

        annotated = box_annotator.annotate(annotated, active_detections)
        annotated = label_annotator.annotate(annotated, active_detections, labels)

        if should_restart_chunk:
            cv2.putText(
                annotated,
                "RF-DETR re-prompt",
                (20, 42),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 255, 255),
                2,
            )

        sink.write_frame(annotated)

        for xyxy, tracker_id in zip(active_detections.xyxy, active_detections.tracker_id):
            x1, y1, x2, y2 = [float(v) for v in xyxy]

            track_rows.append({
                "source_video": str(SOURCE_VIDEO_PATH),
                "frame_idx": int(frame_idx),
                "tracker_id": int(tracker_id),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "center_x": float((x1 + x2) / 2.0),
                "center_y": float((y1 + y2) / 2.0),
                "is_prompt_frame": bool(should_restart_chunk),
            })

tracks_df = pd.DataFrame(track_rows)
prompts_df = pd.DataFrame(prompt_rows)

tracks_df.to_csv(TRACKS_CSV, index=False)
prompts_df.to_csv(PROMPTS_CSV, index=False)

print("raw video:", TARGET_VIDEO_PATH)
print("tracks csv:", TRACKS_CSV)
print("prompts csv:", PROMPTS_CSV)
print("track rows:", len(tracks_df))
print("prompt rows:", len(prompts_df))
print("unique tracker ids:", tracks_df["tracker_id"].nunique() if len(tracks_df) else 0)

display(tracks_df.head())
display(prompts_df.tail())

In [ ]:
# ===== Cell 41 =====
import subprocess
from IPython.display import Video

cmd = [
    "ffmpeg",
    "-y",
    "-loglevel", "error",
    "-i", str(TARGET_VIDEO_PATH),
    "-vcodec", "libx264",
    "-crf", "28",
    str(TARGET_VIDEO_COMPRESSED_PATH),
]

subprocess.run(cmd, check=True)

print("compressed:", TARGET_VIDEO_COMPRESSED_PATH)
Video(str(TARGET_VIDEO_COMPRESSED_PATH), embed=True, width=720)


<details>
<summary>Prototype based Team Classification </summary>

43. Build tighter upper-jersey crops from SAM2 tracks  
44. Build automatic team prototypes from clean crops  
45. Assign teams to crops and smooth by tracker ID  
46. Preview crops grouped by final team  
47. Preview team-colored detections on frames  
48. Render annotated team classification video
49. Preview annotated team classification video


</details>

In [ ]:
# ===== Cell 43 =====

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


if "tracks_df" not in globals():
    TRACKS_CSV = Path("/content/drive/MyDrive/Hockey Analysis Program/sam2_hockey_tracking_review/HockeyUnedited4_seg0021_sam2_test_chunked_reid_sam2_tracks.csv")
    tracks_df = pd.read_csv(TRACKS_CSV)

SOURCE_VIDEO_PATH = Path(tracks_df["source_video"].iloc[0])

print("Video exists:", SOURCE_VIDEO_PATH.exists())
print("Video path:", SOURCE_VIDEO_PATH)
print("Track rows:", len(tracks_df))
print("Unique tracker IDs:", tracks_df["tracker_id"].nunique())

# Settings

SAMPLE_EVERY_N_FRAMES = 5
MAX_CROPS = 1000

MIN_CROP_WIDTH = 20
MIN_CROP_HEIGHT = 35

# Crop helper

def make_team_torso_crop(frame_bgr, x1, y1, x2, y2):
    """
    Create a tighter upper-jersey crop for team-color clustering.

    The goal is to avoid:
    - pants/skates
    - sticks
    - ice
    - boards
    - nearby players

    This focuses mostly on the upper torso/chest/back.
    """
    h, w = frame_bgr.shape[:2]

    x1 = int(max(0, x1))
    y1 = int(max(0, y1))
    x2 = int(min(w, x2))
    y2 = int(min(h, y2))

    box_w = x2 - x1
    box_h = y2 - y1

    if box_w < MIN_CROP_WIDTH or box_h < MIN_CROP_HEIGHT:
        return None

    # Tighter horizontal crop around the center of the player
    # This helps avoid nearby players and sticks.
    cx = (x1 + x2) / 2
    new_w = box_w * 0.45

    tx1 = int(max(0, cx - new_w / 2))
    tx2 = int(min(w, cx + new_w / 2))

    # Upper torso only:
    # skip helmet/top, stop before pants/skates.
    ty1 = int(y1 + 0.18 * box_h)
    ty2 = int(y1 + 0.52 * box_h)

    if tx2 <= tx1 or ty2 <= ty1:
        return None

    crop_bgr = frame_bgr[ty1:ty2, tx1:tx2]

    if crop_bgr.size == 0:
        return None

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    return crop_rgb

# Sample track rows

sample_df = tracks_df[tracks_df["frame_idx"] % SAMPLE_EVERY_N_FRAMES == 0].copy()

if len(sample_df) > MAX_CROPS:
    sample_df = sample_df.sample(MAX_CROPS, random_state=42).sort_values("frame_idx")

print("Sampled track rows:", len(sample_df))

# Build crops

cap = cv2.VideoCapture(str(SOURCE_VIDEO_PATH))

if not cap.isOpened():
    raise ValueError(f"Could not open video: {SOURCE_VIDEO_PATH}")

crops = []
crop_rows = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Building upper-jersey crops"):
    frame_idx = int(row["frame_idx"])

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame_bgr = cap.read()

    if not ok:
        continue

    crop_rgb = make_team_torso_crop(
        frame_bgr,
        row["x1"],
        row["y1"],
        row["x2"],
        row["y2"],
    )

    if crop_rgb is None:
        continue

    crops.append(crop_rgb)

    crop_rows.append({
        "frame_idx": frame_idx,
        "tracker_id": int(row["tracker_id"]),
        "x1": float(row["x1"]),
        "y1": float(row["y1"]),
        "x2": float(row["x2"]),
        "y2": float(row["y2"]),
        "center_x": float((row["x1"] + row["x2"]) / 2.0),
        "center_y": float((row["y1"] + row["y2"]) / 2.0),
    })

cap.release()

crop_df = pd.DataFrame(crop_rows)

print("Created team crops:", len(crops))
print("crop_df rows:", len(crop_df))

if len(crop_df) == 0:
    raise ValueError("No crops were created. Check tracks_df and SOURCE_VIDEO_PATH.")

display(crop_df.head())

In [ ]:
# ===== Cell 44 =====

import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

if "crops" not in globals() or len(crops) == 0:
    raise ValueError("No crops found. Run Cell 43 first.")

if "crop_df" not in globals() or len(crop_df) == 0:
    raise ValueError("No crop_df found. Run Cell 43 first.")


def extract_team_features_and_quality(crop_rgb):
    """
    Extract color/appearance features and a crop quality score.

    The quality score favors crops with:
    - enough colorful jersey pixels
    - not too much ice/white background
    - not too much dark/black area
    - reasonable saturation
    """
    crop_rgb = cv2.resize(crop_rgb, (64, 64))
    hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)

    h = hsv[:, :, 0].astype(np.float32)
    s = hsv[:, :, 1].astype(np.float32)
    v = hsv[:, :, 2].astype(np.float32)

    total_pixels = crop_rgb.shape[0] * crop_rgb.shape[1]

    # Masks
    colorful_mask = (s > 40) & (v > 40)
    white_mask = (s < 35) & (v > 150)
    dark_mask = v < 45

    colorful_ratio = colorful_mask.sum() / total_pixels
    white_ratio = white_mask.sum() / total_pixels
    dark_ratio = dark_mask.sum() / total_pixels

    if colorful_mask.sum() >= 20:
        selected_h = h[colorful_mask]
        selected_s = s[colorful_mask]
        selected_v = v[colorful_mask]
    else:
        selected_h = h.flatten()
        selected_s = s.flatten()
        selected_v = v.flatten()

    # Hue/color ratios. OpenCV hue is 0-179.
    red_ratio = (((selected_h <= 12) | (selected_h >= 160)) & (selected_s > 40)).sum() / max(1, len(selected_h))
    orange_brown_ratio = ((selected_h >= 5) & (selected_h <= 25) & (selected_s > 35)).sum() / max(1, len(selected_h))
    blue_ratio = ((selected_h >= 85) & (selected_h <= 130) & (selected_s > 40)).sum() / max(1, len(selected_h))
    cyan_ratio = ((selected_h >= 70) & (selected_h < 95) & (selected_s > 40)).sum() / max(1, len(selected_h))

    mean_rgb = crop_rgb.reshape(-1, 3).mean(axis=0)
    std_rgb = crop_rgb.reshape(-1, 3).std(axis=0)

    features = [
        float(np.mean(selected_h)),
        float(np.std(selected_h)),
        float(np.mean(selected_s)),
        float(np.std(selected_s)),
        float(np.mean(selected_v)),
        float(np.std(selected_v)),

        float(colorful_ratio),
        float(white_ratio),
        float(dark_ratio),

        float(red_ratio),
        float(orange_brown_ratio),
        float(blue_ratio),
        float(cyan_ratio),

        float(mean_rgb[0]),
        float(mean_rgb[1]),
        float(mean_rgb[2]),
        float(std_rgb[0]),
        float(std_rgb[1]),
        float(std_rgb[2]),
    ]

    # Quality: reward colorful jersey signal, penalize too much ice/white/dark.
    quality = (
        1.8 * colorful_ratio
        + 0.8 * float(np.mean(selected_s) / 255.0)
        - 0.8 * white_ratio
        - 0.5 * dark_ratio
    )

    # Keep in a nicer range
    quality = float(max(0.0, min(1.0, quality)))

    return features, quality


# Extract features and quality scores
feature_rows = []
quality_scores = []

for crop in crops:
    features, quality = extract_team_features_and_quality(crop)
    feature_rows.append(features)
    quality_scores.append(quality)

features = np.array(feature_rows, dtype=np.float32)
quality_scores = np.array(quality_scores, dtype=np.float32)

crop_df["team_crop_quality"] = quality_scores

# Standardize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Select high-quality crops only for building prototypes
QUALITY_PERCENTILE = 60
MIN_QUALITY = np.percentile(quality_scores, QUALITY_PERCENTILE)

prototype_mask = quality_scores >= MIN_QUALITY

# Safety fallback in case too few crops pass
if prototype_mask.sum() < 20:
    print("Warning: few high-quality crops found. Lowering threshold.")
    MIN_QUALITY = np.percentile(quality_scores, 40)
    prototype_mask = quality_scores >= MIN_QUALITY

prototype_features = features_scaled[prototype_mask]

print("Total crops:", len(crops))
print("Prototype candidate crops:", prototype_mask.sum())
print("Quality threshold:", MIN_QUALITY)

# Build 2 automatic team prototypes from only the cleaner crops
prototype_kmeans = KMeans(n_clusters=2, random_state=42, n_init=30)
prototype_labels = prototype_kmeans.fit_predict(prototype_features)

prototype_centers = prototype_kmeans.cluster_centers_

# Store candidate prototype info
crop_df["used_for_team_prototype"] = prototype_mask

# Assign prototype labels back only for prototype candidates
crop_df["prototype_cluster"] = -1
crop_df.loc[prototype_mask, "prototype_cluster"] = prototype_labels

print("\nPrototype cluster counts from clean crops:")
print(pd.Series(prototype_labels).value_counts())

print("\nQuality score summary:")
print(crop_df["team_crop_quality"].describe())

display(crop_df.head())

In [ ]:
# ===== Cell 45 =====

import numpy as np
import pandas as pd
from collections import defaultdict

if "prototype_centers" not in globals():
    raise ValueError("No prototype_centers found. Run Cell 44 first.")

if "features_scaled" not in globals():
    raise ValueError("No features_scaled found. Run Cell 44 first.")

if "tracker_id" not in crop_df.columns:
    raise ValueError("crop_df must include tracker_id. Use SAM2-track crops from Cell 43.")


# Similarity of every crop to each team prototype
sims = cosine_similarity(features_scaled, prototype_centers)

# Raw crop-level assignment
raw_team = sims.argmax(axis=1)

# Confidence = difference between best and second-best prototype similarity
sorted_sims = np.sort(sims, axis=1)
team_margin = sorted_sims[:, -1] - sorted_sims[:, -2]
team_similarity = sorted_sims[:, -1]

crop_df["team_cluster_raw"] = raw_team
crop_df["team_similarity"] = team_similarity
crop_df["team_margin"] = team_margin

# A crop is confident if it is high quality and clearly closer to one prototype
MIN_ASSIGN_QUALITY = 0.20
MIN_ASSIGN_MARGIN = 0.03

crop_df["team_crop_confident"] = (
    (crop_df["team_crop_quality"] >= MIN_ASSIGN_QUALITY) &
    (crop_df["team_margin"] >= MIN_ASSIGN_MARGIN)
)

# Smooth per tracker_id using weighted vote
tracker_team_rows = []

for tracker_id, group in crop_df.groupby("tracker_id"):
    team_scores = defaultdict(float)
    team_counts = defaultdict(int)

    for _, row in group.iterrows():
        team = int(row["team_cluster_raw"])

        # Weight by quality and margin so clean, clear crops matter more
        weight = float(row["team_crop_quality"]) * float(max(row["team_margin"], 0.0))

        # Give a small weight even if margin is tiny, but less influence
        if not bool(row["team_crop_confident"]):
            weight *= 0.25

        team_scores[team] += weight
        team_counts[team] += 1

    if len(team_scores) == 0:
        final_team = -1
        vote_ratio = 0.0
    else:
        sorted_scores = sorted(team_scores.items(), key=lambda x: x[1], reverse=True)
        final_team = int(sorted_scores[0][0])
        best_score = float(sorted_scores[0][1])
        total_score = float(sum(team_scores.values()))

        if total_score > 0:
            vote_ratio = best_score / total_score
        else:
            vote_ratio = 0.0

    tracker_team_rows.append({
        "tracker_id": int(tracker_id),
        "team_cluster_smoothed": final_team,
        "team_vote_ratio": vote_ratio,
        "team_score_0": float(team_scores.get(0, 0.0)),
        "team_score_1": float(team_scores.get(1, 0.0)),
        "team_count_0": int(team_counts.get(0, 0)),
        "team_count_1": int(team_counts.get(1, 0)),
        "num_crops": int(len(group)),
        "num_confident_crops": int(group["team_crop_confident"].sum()),
        "avg_quality": float(group["team_crop_quality"].mean()),
        "avg_margin": float(group["team_margin"].mean()),
    })

tracker_team_df = pd.DataFrame(tracker_team_rows)

# Mark weak tracker-level assignments as uncertain
MIN_TRACK_VOTE_RATIO = 0.60
tracker_team_df["team_cluster_final"] = tracker_team_df["team_cluster_smoothed"]

tracker_team_df.loc[
    tracker_team_df["team_vote_ratio"] < MIN_TRACK_VOTE_RATIO,
    "team_cluster_final"
] = -1

# Merge final tracker-level team back into crop_df
for col in ["team_cluster_smoothed", "team_vote_ratio", "team_cluster_final"]:
    if col in crop_df.columns:
        crop_df = crop_df.drop(columns=[col])

crop_df = crop_df.merge(
    tracker_team_df[["tracker_id", "team_cluster_smoothed", "team_vote_ratio", "team_cluster_final"]],
    on="tracker_id",
    how="left"
)

print("Tracker-level team assignments:")
display(tracker_team_df.sort_values("team_vote_ratio"))

print("\nFinal team counts by tracker:")
print(tracker_team_df["team_cluster_final"].value_counts())

print("\nUpdated crop_df:")
display(crop_df.head())

In [ ]:
# ===== Cell 46 =====

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import random

if "team_cluster_final" in crop_df.columns:
    team_col = "team_cluster_final"
elif "team_cluster_smoothed" in crop_df.columns:
    team_col = "team_cluster_smoothed"
elif "team_cluster_raw" in crop_df.columns:
    team_col = "team_cluster_raw"
else:
    raise ValueError("No team assignment column found. Run Cells 44 and 45 first.")

print("Using team column:", team_col)

for team_id in sorted(crop_df[team_col].dropna().unique()):
    team_id = int(team_id)

    if team_id == -1:
        title = "Uncertain team"
    else:
        title = f"Team cluster {team_id}"

    team_rows = crop_df[crop_df[team_col] == team_id].copy()

    if len(team_rows) == 0:
        continue

    # Prefer higher quality examples for preview
    team_rows = team_rows.sort_values("team_crop_quality", ascending=False).head(50)

    print(title, "| crops:", len(team_rows))

    plt.figure(figsize=(14, 7))

    for i, idx in enumerate(team_rows.index[:50]):
        crop = crops[idx]

        plt.subplot(5, 10, i + 1)
        plt.imshow(crop)
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# ===== Cell 47 =====

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Choose final team column
if "team_cluster_final" in crop_df.columns:
    team_col = "team_cluster_final"
elif "team_cluster_smoothed" in crop_df.columns:
    team_col = "team_cluster_smoothed"
elif "team_cluster_raw" in crop_df.columns:
    team_col = "team_cluster_raw"
else:
    raise ValueError("No team labels found. Run Cells 44 and 45 first.")

# Automatically pick a frame with the most player crops
PREVIEW_FRAME_IDX = int(crop_df["frame_idx"].value_counts().idxmax())

# Or manually choose one:
# PREVIEW_FRAME_IDX = 180

SOURCE_VIDEO_PATH = Path(tracks_df["source_video"].iloc[0])

print("Preview frame:", PREVIEW_FRAME_IDX)
print("Source video:", SOURCE_VIDEO_PATH)
print("Video exists:", SOURCE_VIDEO_PATH.exists())
print("Using team column:", team_col)

cap = cv2.VideoCapture(str(SOURCE_VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, PREVIEW_FRAME_IDX)

ok, frame_bgr = cap.read()
cap.release()

if not ok:
    raise ValueError(f"Could not read frame {PREVIEW_FRAME_IDX}")

preview = frame_bgr.copy()

frame_crops = crop_df[crop_df["frame_idx"] == PREVIEW_FRAME_IDX].copy()

print("Players/crops on this frame:", len(frame_crops))

if len(frame_crops) == 0:
    raise ValueError(f"No crops found for frame {PREVIEW_FRAME_IDX}.")

# BGR colors
cluster_colors = {
    0: (255, 0, 0),      # blue
    1: (0, 0, 255),      # red
    -1: (0, 255, 255),   # yellow = uncertain
}

for _, row in frame_crops.iterrows():
    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    team_cluster = int(row[team_col])
    color = cluster_colors.get(team_cluster, (0, 255, 0))

    if team_cluster == -1:
        team_text = "uncertain"
    else:
        team_text = f"team {team_cluster}"

    if "tracker_id" in row:
        label = f"id {int(row['tracker_id'])} | {team_text}"
    else:
        label = team_text

    if "team_vote_ratio" in row and not np.isnan(row["team_vote_ratio"]):
        label += f" | {row['team_vote_ratio']:.2f}"

    cv2.rectangle(preview, (x1, y1), (x2, y2), color, 3)

    cv2.putText(
        preview,
        label,
        (x1, max(20, y1 - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        color,
        2,
    )

preview_rgb = cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15, 9))
plt.imshow(preview_rgb)
plt.axis("off")
plt.title(f"Prototype team preview — frame {PREVIEW_FRAME_IDX}")
plt.show()

In [ ]:
# ===== Cell 48 =====

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import subprocess

# Output paths

DEMO_OUT_DIR = Path("/content/drive/MyDrive/Hockey Analysis Program/demo_outputs")
DEMO_OUT_DIR.mkdir(parents=True, exist_ok=True)

DEMO_RAW_OUT = DEMO_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_annotated_raw.mp4"
DEMO_BROWSER_OUT = DEMO_OUT_DIR / f"{SOURCE_VIDEO_PATH.stem}_annotated_demo.mp4"

print("Source video:", SOURCE_VIDEO_PATH)
print("Raw output:", DEMO_RAW_OUT)
print("Browser-friendly output:", DEMO_BROWSER_OUT)

# Check required data

if "tracks_df" not in globals():
    raise ValueError("tracks_df not found. Run SAM2 tracking first.")

if "crop_df" not in globals():
    raise ValueError("crop_df not found. Run team classification first.")

if "team_cluster_final" not in crop_df.columns:
    raise ValueError("team_cluster_final not found. Run prototype team classification cells first.")

# Build tracker_id -> team map

tracker_team_df = (
    crop_df[["tracker_id", "team_cluster_final", "team_vote_ratio"]]
    .drop_duplicates(subset=["tracker_id"])
    .copy()
)

team_map = {}
team_conf_map = {}

for _, row in tracker_team_df.iterrows():
    tracker_id = int(row["tracker_id"])
    team_map[tracker_id] = int(row["team_cluster_final"])
    team_conf_map[tracker_id] = float(row["team_vote_ratio"])

print("Tracker team assignments:")
display(tracker_team_df.sort_values("tracker_id"))

# Open source video

cap = cv2.VideoCapture(str(SOURCE_VIDEO_PATH))

if not cap.isOpened():
    raise ValueError(f"Could not open source video: {SOURCE_VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Width/Height:", width, height)
print("Total frames:", total_frames)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(DEMO_RAW_OUT), fourcc, fps, (width, height))

# Group tracks by frame for fast lookup
tracks_by_frame = {
    int(frame_idx): group.copy()
    for frame_idx, group in tracks_df.groupby("frame_idx")
}

# BGR colors
team_colors = {
    0: (255, 0, 0),      # blue
    1: (0, 0, 255),      # red
    -1: (0, 255, 255),   # yellow = uncertain
}

# Render annotated video

for frame_idx in tqdm(range(total_frames), desc="Rendering annotated demo"):
    ok, frame_bgr = cap.read()

    if not ok:
        break

    frame_tracks = tracks_by_frame.get(frame_idx, None)

    if frame_tracks is not None:
        for _, row in frame_tracks.iterrows():
            tracker_id = int(row["tracker_id"])

            team_id = team_map.get(tracker_id, -1)
            team_conf = team_conf_map.get(tracker_id, 0.0)

            color = team_colors.get(team_id, (0, 255, 0))

            x1 = int(row["x1"])
            y1 = int(row["y1"])
            x2 = int(row["x2"])
            y2 = int(row["y2"])

            if team_id == -1:
                label = f"ID {tracker_id} | uncertain"
            else:
                label = f"ID {tracker_id} | Team {team_id}"

            label += f" | {team_conf:.2f}"

            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 3)

            cv2.putText(
                frame_bgr,
                label,
                (x1, max(24, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2,
            )

    writer.write(frame_bgr)

cap.release()
writer.release()

print("Saved raw annotated video:")
print(DEMO_RAW_OUT)

cmd = [
    "ffmpeg",
    "-y",
    "-i", str(DEMO_RAW_OUT),
    "-vcodec", "libx264",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    str(DEMO_BROWSER_OUT),
]

subprocess.run(cmd, check=True)

print("Saved browser-friendly demo video:")
print(DEMO_BROWSER_OUT)

In [ ]:
# ===== Cell 49 =====

from IPython.display import Video, display

display(Video(str(DEMO_BROWSER_OUT), embed=True, width=900))